# Adaptive Kalman controller: tutorial

This notebook walks through the new pieces of `kalman/Kalman_tools.py` and the new
`kalman/controller.py`, first on small synthetic data (CPU, a few seconds per cell),
then in the closed loop with the pyramid WFS from `SimpleKalman.ipynb` (GPU, long).

What changed compared to `SimpleKalman.ipynb`:

| | `SimpleKalman.ipynb` | new |
|---|---|---|
| **A** | least squares, biased towards 0 by WFS noise | instrumental variable, from lag covariances (§1.2) |
| **R** | photon + read-out-noise model | estimated from telemetry, no noise model needed (§1.3) |
| **Q, R structure** | full | full, block (Zernike radial order) or diagonal (§1.5) |
| **Gain** | `kalman_filter` updates P every frame ($O(n^3)$) | steady-state gain from the DARE (§1.6) |
| **Real-time step** | several matrix products | one matrix-vector product (§1.7) |
| **Model** | identified once, offline | re-identified every `update_period` frames, validated, swapped between frames (§2) |
| **Monitoring** | none | normalized innovation squared (NIS), optional integrator fallback (§1.8, §2.2) |
| **Vibrations** | AR(1) only | AR(2) for selected modes, e.g. tip/tilt (§1.11, §2.3) |

**Contents**

1. Building blocks on a toy atmosphere
2. `AdaptiveKalmanController` in a toy closed loop
3. Closed loop with the pyramid WFS (CUDA)

Parts 1 and 2 only need `torch` and run on CPU in float64. Part 3 needs `AI4AO` and a GPU.

In [ ]:
import math
import time

import torch
import pylab as plt

from kalman.Kalman_tools import *
from kalman.controller import AdaptiveKalmanConfig, AdaptiveKalmanController, IntegratorController

DTYPE = torch.float64    # parts 1-2: CPU, float64

# one color per controller, used in every plot
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"   # Kalman, integrator, reference / fixed model
plt.rcParams.update({
    "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
    "axes.spines.right": False, "lines.linewidth": 1.5, "figure.figsize": (8, 3.5),
})


def rolling(v, w=200):
    """Moving average of a 1D sequence, returns (frames, values)."""
    v = torch.as_tensor(v, dtype=torch.float64).reshape(1, 1, -1)
    return torch.arange(w - 1, v.shape[-1]), torch.nn.functional.avg_pool1d(v, w, stride=1).flatten()

---
# Part 1: Building blocks on a toy atmosphere

## 1.1 A toy atmosphere with a known model

Independent AR(1) modes observed with white WFS noise:

$$\phi_{k+1} = A\,\phi_k + w_k,\quad w_k \sim \mathcal N(0, Q), \qquad x_k = \phi_k + n_k,\quad n_k \sim \mathcal N(0, R).$$

$x_k$ is the pseudo-open-loop telemetry. In the closed loop it is
$y_k = z_k + u_{k-2}$ (WFS measurement plus the command that was on the DM), which
`estimate_atmospheric_state` computes. Here we generate it directly, so that we know the true $A$, $Q$ and $R$.

The six modes go from slow ($a = 0.995$, like low-order Zernikes) to fast ($a = 0.8$),
with the same noise, so the fast modes have a low SNR.

In [ ]:
def ar1_modes(n_frames, a, q_std, r_std, seed=0, q_change=None):
    """
    Independent AR(1) modes phi_{k+1} = a * phi_k + w_k, and white WFS noise n_k.

    q_change = (frame, factor) multiplies the process noise from that frame on
    (e.g. the seeing degrades). Returns phi, noise, each [n_frames, n_modes].
    """
    g = torch.Generator().manual_seed(seed)
    a = torch.as_tensor(a, dtype=DTYPE)
    w = q_std * torch.randn(n_frames, a.numel(), generator=g, dtype=DTYPE)
    if q_change is not None:
        w[q_change[0]:] *= q_change[1]
    phi = torch.zeros(n_frames, a.numel(), dtype=DTYPE)
    for k in range(n_frames - 1):
        phi[k + 1] = a * phi[k] + w[k]
    noise = r_std * torch.randn(n_frames, a.numel(), generator=g, dtype=DTYPE)
    return phi, noise


a_true = torch.tensor([0.995, 0.99, 0.98, 0.95, 0.9, 0.8], dtype=DTYPE)
q_std, r_std = 0.1, 0.3
phi, noise = ar1_modes(20000, a_true, q_std, r_std)
x = phi + noise                       # pseudo-open-loop telemetry, [N_frames, N_modes]
n = x.shape[1]

signal_var = q_std**2 / (1 - a_true**2)
print("mode         ", list(range(n)))
print("a            ", a_true.tolist())
print("SNR (var/var)", [round(v, 2) for v in (signal_var / r_std**2).tolist()])

## 1.2 Estimating A: least squares vs instrumental variable

Every estimator now works on the **lag covariances** $C_l = E[x_{k+l}\,x_k^T]$
(`lagged_covariances`), so it works the same with a fixed window or with a forgetting factor (§1.10).

* Least squares: $A = C_1 C_0^{-1}$. But $C_0 = \Sigma + R$, so WFS noise biases $A$ towards 0.
* Instrumental variable: $A = C_2 C_1^{-1}$. White noise does not enter lags $\ge 1$
  ($C_1 = A\Sigma$, $C_2 = A^2\Sigma$), so there is no bias and no need to know $R$.

In [ ]:
C0, C1, C2 = lagged_covariances(x, max_lag=2)

A_ls = estimate_A_from_covariances(C0, C1, method="least_squares")
A_iv = estimate_A_from_covariances(C0, C1, C2, method="instrumental_variable")

plt.plot(a_true, "k--", label="true")
plt.plot(torch.diagonal(A_ls), "o-", color=ORANGE, label="least squares")
plt.plot(torch.diagonal(A_iv), "o-", color=BLUE, label="instrumental variable")
plt.xlabel("mode"); plt.ylabel("diagonal of A"); plt.legend(); plt.title("Estimated A (20000 frames)")
plt.show()

Least squares underestimates $a$ badly for the low-SNR modes (e.g. 0.19 instead of 0.8).
The instrumental variable is unbiased, but has more variance for fast low-SNR modes, where
$C_1$ is small. Those modes carry little predictable signal anyway.

## 1.3 Estimating R from telemetry

No photon-noise model is needed (so this also works for a CNN reconstructor, as long as its error is white in time).
`estimate_R_from_covariances` has two forms:

* lag-1: $R = C_0 - A^{-1} C_1$
* residual (with `C2`, the default in `identify_model`): the prediction error
  $e_k = x_{k+1} - A x_k$ has lag-1 covariance $-A R$, so $R = -A^{-1} E[e_k e_{k-1}^T]$.

Both agree when $A$ is exactly the instrumental-variable estimate. They differ when $A$ is
modified, e.g. **shrunk** so that its spectral radius stays below 1 (`limit_spectral_radius`,
which the controller does by default). Here we shrink it to 0.97, which is much more than in practice:

In [ ]:
A_diag = estimate_A_from_covariances(C0, C1, C2, method="instrumental_variable", structure="diagonal")
A_shrunk = limit_spectral_radius(A_diag, max_radius=0.97, structure="diagonal")

R_lag1 = estimate_R_from_covariances(C0, C1, A_shrunk, structure="diagonal")
R_res = estimate_R_from_covariances(C0, C1, A_shrunk, C2=C2, structure="diagonal")

print("A shrunk:", [round(v, 3) for v in torch.diagonal(A_shrunk).tolist()])
plt.axhline(r_std**2, color="k", ls="--", label="true")
plt.plot(torch.diagonal(R_lag1), "o-", color=ORANGE, label="lag-1 form")
plt.plot(torch.diagonal(R_res), "o-", color=BLUE, label="residual form")
plt.xlabel("mode"); plt.ylabel("diagonal of R"); plt.legend(); plt.title("R from telemetry, with A shrunk to 0.97")
plt.show()

For the slow modes (the ones whose $a$ was shrunk), the lag-1 form is off by up to 30%,
while the residual form stays within a few percent.

## 1.4 The whole identification: `identify_model`

`identify_model(C, ...)` chains the steps: $A$ (optionally shrunk), $R$ (residual form, unless given),
optional AR(2) modes, then $Q = Q_{raw} - R - A R A^T$ with negative eigenvalues clipped.
It returns an `IdentifiedModel(A, H, Q, R, Q_unclipped, A_ar1)`.

In [ ]:
model = identify_model(lagged_covariances(x, max_lag=2), A_structure="diagonal")

print(f"{'mode':>4} {'a true':>7} {'a est':>7} {'q true':>7} {'q est':>7} {'r true':>7} {'r est':>7}")
for i in range(n):
    print(f"{i:>4} {a_true[i]:7.3f} {model.A[i, i]:7.3f} {q_std**2:7.4f} {model.Q[i, i]:7.4f} "
          f"{r_std**2:7.4f} {model.R[i, i]:7.4f}")

## 1.5 Structure of A, Q, R, and the spectral radius

With short windows, the off-diagonal entries of a full estimate are mostly noise, and through $K$
they couple modes that are independent. `structure` restricts $A$ (and, with `"match"`, also $Q$ and $R$):

* `"full"`: dense
* `"block"`: one block per group, e.g. Zernike radial order from `zernike_radial_groups`
* `"diagonal"`: independent modes

In this toy the true $A$ is diagonal, so everything off the diagonal is estimation noise:

In [ ]:
groups = zernike_radial_groups(n)          # Noll modes 2..7 -> radial orders
print("radial order of each mode:", groups.tolist())

Cs = lagged_covariances(x[:2000], max_lag=2)    # a short window, as in the controller
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, s in zip(axes, ["full", "block", "diagonal"]):
    A_s = estimate_A_from_covariances(*Cs, method="instrumental_variable", structure=s, groups=groups)
    im = ax.imshow(A_s - torch.diag(a_true), cmap="RdBu_r", vmin=-0.3, vmax=0.3)
    ax.set_title(f'structure="{s}"'); ax.grid(False)
fig.colorbar(im, ax=axes, label="A estimated - A true")
plt.show()

Short windows also make the estimate of a slow mode overshoot $a = 1$ regularly. Rather than rejecting
the whole model, the controller shrinks $A$ to `A_max_radius` (default 0.9995) with `limit_spectral_radius`:

In [ ]:
a0 = torch.stack([
    estimate_A_from_covariances(*lagged_covariances(x[s:s + 500], 2), method="instrumental_variable",
                                structure="diagonal")[0, 0]
    for s in range(0, 20000, 500)
])
print(f"a_0 on 40 windows of 500 frames: {100 * (a0 > 1).float().mean():.0f}% above 1, max {a0.max():.4f}")

A_over = torch.diag(torch.tensor([1.002, 0.99, 0.5], dtype=DTYPE))
print("limited:", torch.diagonal(limit_spectral_radius(A_over, 0.9995, structure="diagonal")).tolist())

## 1.6 Steady-state gain: `steady_state_gain`

The gain of `kalman_filter` converges to a constant $K$, given by the discrete algebraic Riccati equation (DARE):

$$P = A P A^T - A P H^T (H P H^T + R)^{-1} H P A^T + Q, \qquad K = P H^T (H P H^T + R)^{-1}.$$

`steady_state_gain` solves it with a doubling algorithm (iteration $k$ = $2^k$ Riccati steps), so it converges
in a few tens of iterations even for $a \approx 1$. It returns `DAREResult(K, P, converged, iterations, residual)`;
`P` is the predicted covariance $P_{k|k-1}$.

In [ ]:
A, H, Q, R = model.A, model.H, model.Q, model.R

t0 = time.perf_counter()
dare = steady_state_gain(A, H, Q, R, compute_dtype=DTYPE)
t_doubling = time.perf_counter() - t0
t0 = time.perf_counter()
dare_riccati = steady_state_gain(A, H, Q, R, method="riccati", compute_dtype=DTYPE)
t_riccati = time.perf_counter() - t0
print(f"doubling: {dare.iterations:4d} iterations, {1e3 * t_doubling:6.1f} ms, residual {dare.residual:.1e}")
print(f"riccati : {dare_riccati.iterations:4d} iterations, {1e3 * t_riccati:6.1f} ms, residual {dare_riccati.residual:.1e}")

# the time-varying filter of SimpleKalman.ipynb converges to the same gain
state = KalmanState(x=torch.zeros(n, dtype=DTYPE), P=Q.clone())
matrices = KalmanMatrices(A=A, H=H, Q=Q, R=R)
err = []
for k in range(200):
    state, K = kalman_filter(x[k], state, matrices)
    err.append(float(torch.linalg.norm(K - dare.K) / torch.linalg.norm(dare.K)))

plt.semilogy(err, color=BLUE)
plt.xlabel("frame"); plt.ylabel(r"$\|K_k - K_\infty\| / \|K_\infty\|$")
plt.title("kalman_filter gain vs steady_state_gain"); plt.show()

## 1.7 The real-time step as one matrix-vector product

With a constant $K$, the filter update and the command prediction are linear in
$[x_{k-1|k-1};\ y_k]$, so `build_realtime_matrix` stacks them into one matrix $M$:

$$
\begin{bmatrix} x_{k|k} \\ u_k \end{bmatrix} = M \begin{bmatrix} x_{k-1|k-1} \\ y_k \end{bmatrix},
\qquad
M = \begin{bmatrix} F & K \\ C A^d F & C A^d K \end{bmatrix}, \quad F = (I - KH)A .
$$

With $y_k = z_k + u_{k-2}$, $x_{k|k}$ estimates $\phi_{k-1}$ and $u_k$ is on the DM during frame $k+1$,
so the command is predicted $d = 2$ steps ahead. `realtime_step` applies $M$ (also batched: `[B, N]` inputs).

In [ ]:
d = 2
M = build_realtime_matrix(A, H, dare.K, d)
print("M:", tuple(M.shape), "= [N_state + N_cmd, N_state + N_meas]")

# same thing, step by step
Ad = torch.linalg.matrix_power(A, d)
x_post = torch.zeros(n, dtype=DTYPE)
x_rt = torch.zeros(n, dtype=DTYPE)
max_diff = 0.0
for k in range(500):
    x_pred = A @ x_post
    x_post = x_pred + dare.K @ (x[k] - H @ x_pred)
    u_ref = H @ Ad @ x_post
    x_rt, u_rt = realtime_step(M, x_rt, x[k])
    max_diff = max(max_diff, float((u_rt - u_ref).abs().max()))
print(f"max |u_realtime - u_step_by_step| over 500 frames: {max_diff:.1e}")

plt.figure(figsize=(4, 4))
plt.imshow(M, cmap="RdBu_r", vmin=-M.abs().max(), vmax=M.abs().max()); plt.grid(False)
plt.axhline(n - 0.5, color="k", lw=1); plt.axvline(n - 0.5, color="k", lw=1)
plt.xticks([n / 2 - 0.5, 3 * n / 2 - 0.5], [r"$x_{k-1|k-1}$", r"$y_k$"])
plt.yticks([n / 2 - 0.5, 3 * n / 2 - 0.5], [r"$x_{k|k}$", r"$u_k$"])
plt.title("M"); plt.show()

## 1.8 Is the model consistent with the data? The NIS

The innovation $\nu_k = y_k - H A x_{k-1|k-1}$ has covariance $S = H P H^T + R$ if the model is right.
`innovation_whitener` returns $W = L^{-1}$ ($S = L L^T$), so $e_k = W\nu_k$ has identity covariance and the
normalized innovation squared $\text{NIS}_k = \|e_k\|^2$ has mean $N_{meas}$.

Passing `whiten=W` to `build_realtime_matrix` adds $e_k$ as extra output rows, so the NIS costs nothing extra in real time.
Below, the true model vs a model that assumes 10 times less WFS noise than there is:

In [ ]:
def nis_trace(M, y, n_state, n_cmd):
    """NIS / N_meas of the whitened-innovation rows of M, frame by frame."""
    xk = torch.zeros(n_state, dtype=DTYPE)
    out = []
    for yk in y:
        xk, ue = realtime_step(M, xk, yk)
        e = ue[n_cmd:]
        out.append(float(e @ e) / e.numel())
    return out


W = innovation_whitener(H, dare.P, R)
M_w = build_realtime_matrix(A, H, dare.K, d, whiten=W)
print("M with whitening rows:", tuple(M_w.shape))

R_wrong = R / 10
dare_wrong = steady_state_gain(A, H, Q, R_wrong, compute_dtype=DTYPE)
M_wrong = build_realtime_matrix(A, H, dare_wrong.K, d, whiten=innovation_whitener(H, dare_wrong.P, R_wrong))

for Mi, color, label in [(M_w, BLUE, "identified model"), (M_wrong, ORANGE, "R 10x too small")]:
    plt.plot(*rolling(nis_trace(Mi, x[:5000], n, n)), color=color, label=label)
plt.axhline(1, color="k", ls="--", lw=1)
plt.xlabel("frame"); plt.ylabel("NIS / N_meas (200-frame mean)"); plt.legend()
plt.title("Innovation consistency"); plt.show()

## 1.9 Validating a model before using it

`validate_model(A, Q_unclipped, R, dare)` checks that everything is finite, $\rho(A)$ is below
`max_spectral_radius`, $Q$ was not too negative before clipping, $R$ is positive definite and the DARE
converged. It returns a `ModelCheck`. A negative $Q$ typically means $R$ is overestimated, or
the window is too short to separate the atmosphere from the noise.

In [ ]:
C = lagged_covariances(x, max_lag=2)
print("identified model :", validate_model(model.A, model.Q_unclipped, model.R, dare))

A_unstable = model.A.clone(); A_unstable[0, 0] = 1.001
print("unstable A       :", validate_model(A_unstable, model.Q_unclipped, model.R).reasons)

model_bad_R = identify_model(C, A_structure="diagonal", R=3 * r_std**2 * torch.eye(n, dtype=DTYPE))
print("R 3x too large   :", validate_model(model_bad_R.A, model_bad_R.Q_unclipped, model_bad_R.R).reasons)

## 1.10 Forgetting factor: `RecursiveCovariance`

Instead of a fixed window, the lag covariances can be updated with a forgetting factor $\lambda$:

$$S_l \leftarrow \lambda S_l + (1-\lambda)\, x_k x_{k-l}^T, \qquad C_l = S_l / w,$$

with memory $\approx 1/(1-\lambda)$ frames and $w$ correcting the start-up bias. Frames can be fed one at a
time or in blocks, with the same result. It needs no telemetry buffer, and the memory is a continuous knob.
Below, the process noise doubles at frame 6000. With similar memories, both estimates follow the change with a
similar delay (the slowest mode itself takes a few hundred frames to reach its new variance):

In [ ]:
a4 = a_true[:4]
phi_c, noise_c = ar1_modes(12000, a4, q_std, r_std, seed=1, q_change=(6000, 2.0))
x_c = phi_c + noise_c

rc = RecursiveCovariance(4, forgetting=1 - 1 / 1000, max_lag=2, dtype=DTYPE)
frames, rec, win = [], [], []
for k in range(100, 12001, 100):
    rc.update(x_c[k - 100:k])                 # blocks of 100 frames
    frames.append(k)
    rec.append(float(torch.diagonal(rc.covariances()[0]).mean()))
    win.append(float(torch.diagonal(lagged_covariances(x_c[max(0, k - 2000):k], 0)[0]).mean()))

true_var = [float((q_std**2 * (1 if k < 6000 else 4) / (1 - a4**2)).mean() + r_std**2) for k in frames]
plt.plot(frames, true_var, "k--", label="true")
plt.plot(frames, win, color=ORANGE, label="window of 2000 frames")
plt.plot(frames, rec, color=BLUE, label=r"forgetting factor, $1/(1-\lambda)$ = 1000")
plt.xlabel("frame"); plt.ylabel(r"mean diagonal of $C_0$"); plt.legend(); plt.title("Tracking a change")
plt.show()

rc_block = RecursiveCovariance(4, 0.999, dtype=DTYPE); rc_block.update(x_c[:300])
rc_frame = RecursiveCovariance(4, 0.999, dtype=DTYPE)
for row in x_c[:300]:
    rc_frame.update(row)
diff = max(float((a - b).abs().max()) for a, b in zip(rc_block.covariances(), rc_frame.covariances()))
print(f"block vs frame-by-frame updates: max difference {diff:.1e}")

## 1.11 Vibrations: AR(2) modes

A vibration at frequency $f$ (sampling $f_s$) with pole radius $\rho$ is an AR(2) process:

$$x_{k+1} = a_1 x_k + a_2 x_{k-1} + w_k, \qquad a_1 = 2\rho\cos(2\pi f/f_s),\quad a_2 = -\rho^2 .$$

An AR(1) model cannot predict an oscillation. For the modes listed in `ar2_modes`, the state is augmented
with their previous value, $X_k = [x_k;\ x_{k-1}[S]]$ (`augment_ar2`), and $a_1, a_2$ come from the
autocovariances. As for $A$, least squares is biased by noise, and the instrumental variable (overdetermined
Yule-Walker up to lag `max_lag`) is not.

In [ ]:
def vibration(n_frames, f, rho, q, fs=1000, seed=1):
    """AR(2) resonance at f Hz (pole radius rho), driven by white noise of std q."""
    g = torch.Generator().manual_seed(seed)
    a1, a2 = 2 * rho * math.cos(2 * math.pi * f / fs), -rho**2
    v = torch.zeros(n_frames, dtype=DTYPE)
    w = q * torch.randn(n_frames, generator=g, dtype=DTYPE)
    for k in range(1, n_frames - 1):
        v[k + 1] = a1 * v[k] + a2 * v[k - 1] + w[k]
    return v


fs, f_vib, rho = 1000, 40, 0.99
v = vibration(20000, f_vib, rho, q=0.05)
xv = torch.stack([v, phi[:, 1]], dim=1) + 0.1 * torch.randn(20000, 2, dtype=DTYPE)

print(f"true: a1 = {2 * rho * math.cos(2 * math.pi * f_vib / fs):.4f}, a2 = {-rho**2:.4f}, f = {f_vib} Hz")
for method in ["least_squares", "instrumental_variable"]:
    a1, a2 = estimate_ar2(xv, modes=[0], method=method, max_lag=20)
    f_est = fs / (2 * math.pi) * math.acos(float(a1) / (2 * math.sqrt(-float(a2))))
    print(f"{method:>22}: a1 = {float(a1):.4f}, a2 = {float(a2):.4f}, f = {f_est:.1f} Hz, rho = {math.sqrt(-float(a2)):.4f}")

model_ar2 = identify_model(lagged_covariances(xv, 2), A_structure="diagonal",
                           ar2_modes=[0], autocov=autocovariances(xv, 20))
print("\naugmented A (state [x_0, x_1, x_0 previous]):\n", model_ar2.A.numpy().round(4))
print("H:\n", model_ar2.H.numpy())

seg = xv[:, 0].reshape(-1, 1000)
psd = (torch.fft.rfft(seg - seg.mean(1, keepdim=True)).abs() ** 2).mean(0)
plt.semilogy(torch.fft.rfftfreq(1000, 1 / fs), psd, color=BLUE)
plt.axvline(f_vib, color="k", ls="--", lw=1)
plt.xlabel("frequency (Hz)"); plt.ylabel("PSD"); plt.title("Telemetry of the vibrating mode"); plt.show()

> **Caveat: lightly damped vibrations.** With $\rho \gtrsim 0.998$ and 2000-frame windows, the $Q$ estimated
> for that mode can come out $\le 0$. It is then clipped to 0, the gain for the mode becomes 0, and the
> filter stops correcting it. `validate_model` does not catch this, because it looks at the negative part
> of the whole spectrum of $Q$. The same problem shows up for AR(1) modes in §2.4.

---
# Part 2: `AdaptiveKalmanController` in a toy closed loop

The controller wraps everything from Part 1:

* **Real-time path** (`rt_step`, every frame, no host synchronization):
  $y_k = z_k + u_{k-\text{pol\_lag}}$, one product with the double-buffered matrix $M$,
  the integrator command $u^I_k$ (always computed), and $u_k = \alpha\, u^{KF}_k + (1-\alpha)\, u^I_k$.
* **Slow path** (`slow_path`, between frames): every `update_period` frames, identify a model from
  the last `window` frames of telemetry, solve the DARE, validate it, and write $M$ into the *inactive*
  buffer before flipping the index. The NIS is checked every `nis_window` frames.
* **Bootstrap**: until the first model is accepted, the integrator runs (to collect telemetry).

`step(z)` = `rt_step(z)` + `slow_path()`.

The loop below has the timing of `SimpleKalman.ipynb`. During frame $k$ the DM holds $u_{k-1}$, and the
WFS reads the residual of frame $k-1$:

$$\text{residual}_k = \phi_k - u_{k-1}, \qquad z_k = \phi_{k-1} - u_{k-2} + n_k,$$

so $y_k = z_k + u_{k-2} = \phi_{k-1} + n_k$ (`delay=2`), and $u_k$ must predict $\phi_{k+1}$ (`command_lead=2`).

In [ ]:
def run_loop(ctrl, phi, noise, start=2, stop=None):
    """Closed loop with a 2-frame delay. Returns the residuals [frames, N_modes]."""
    stop = phi.shape[0] if stop is None else stop
    u1 = torch.zeros(phi.shape[1], dtype=DTYPE)   # u_{k-1}
    u2 = torch.zeros(phi.shape[1], dtype=DTYPE)   # u_{k-2}
    res = []
    for k in range(start, stop):
        res.append(phi[k] - u1)                   # the DM holds u_{k-1} during frame k
        z = phi[k - 1] - u2 + noise[k]
        u = ctrl.step(z).clone()                  # step returns an internal buffer: clone it
        u2, u1 = u1, u
    return torch.stack(res)


def toy_config(**kw):
    base = dict(n_modes=4, delay=2, window=2000, update_period=1000, A_structure="diagonal",
                integrator_gain=0.4, integrator_leak=0.99, nis_window=200,
                ident_dtype=DTYPE, dare_dtype=DTYPE)
    base.update(kw)
    return AdaptiveKalmanConfig(**base)


def show_history(ctrl, skip=()):
    for h in ctrl.history:
        if h["event"] in skip:
            continue
        extra = ", ".join(f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}"
                          for k, v in h.items() if k not in ("frame", "event"))
        print(f"{h['frame']:6d}  {h['event']:<10} {extra}")


def event_lines(ctrl, event="accepted", **kw):
    for h in ctrl.history:
        if h["event"] == event:
            plt.axvline(h["frame"], color="0.6", ls=":", lw=1, **kw)

## 2.1 Bootstrap, first model, re-identification

Four slow modes, compared with the integrator and with a Kalman filter that knows the true model from frame 0
(the best achievable). Until frame 2000 the adaptive controller runs the integrator, so its curve hides the integrator's.

In [ ]:
A_SLOW = [0.995, 0.99, 0.98, 0.95]
phi2, noise2 = ar1_modes(8000, A_SLOW, 0.1, 0.3, seed=0)


def true_model_controller(a, q, r):
    """Controller running the true steady-state model from frame 0 (no re-identification)."""
    A = torch.diag(torch.as_tensor(a, dtype=DTYPE))
    I = torch.eye(A.shape[0], dtype=DTYPE)
    dare = steady_state_gain(A, I, q**2 * I, r**2 * I, compute_dtype=DTYPE)
    ctrl = AdaptiveKalmanController(toy_config(max_updates=0), "cpu", DTYPE)
    ctrl.load_matrix(build_realtime_matrix(A, I, dare.K, 2, whiten=innovation_whitener(I, dare.P, r**2 * I)))
    return ctrl


ctrl = AdaptiveKalmanController(toy_config(), "cpu", DTYPE)
res_kf = run_loop(ctrl, phi2, noise2)
res_int = run_loop(IntegratorController(4, gain=0.4, leak=0.99, dtype=DTYPE), phi2, noise2)
res_true = run_loop(true_model_controller(A_SLOW, 0.1, 0.3), phi2, noise2)

print("identical to the integrator before the first model:", torch.allclose(res_kf[:1990], res_int[:1990]))
print("mode:", ctrl.mode, "| accepted models:", ctrl.n_accepted, "\n")
show_history(ctrl)

for res, color, label in [(res_int, ORANGE, "integrator"), (res_true, AQUA, "Kalman, true model"),
                          (res_kf, BLUE, "adaptive Kalman")]:
    plt.plot(*rolling(res.pow(2).mean(1)), color=color, label=label)
event_lines(ctrl)
plt.xlabel("frame"); plt.ylabel("residual variance (200-frame mean)"); plt.legend()
plt.title("Toy closed loop (dotted: accepted models)"); plt.show()

for name, res in [("integrator", res_int), ("adaptive Kalman", res_kf), ("true model", res_true)]:
    print(f"{name:>16}: residual variance after frame 3000 = {float(res[3000:].pow(2).mean()):.4f}")

After each run, the controller keeps the last accepted model, its DARE solution and its check, and the NIS log:

In [ ]:
print("model.A diagonal:", [round(v, 4) for v in torch.diagonal(ctrl.model.A).tolist()])
print("dare.K diagonal :", [round(v, 3) for v in torch.diagonal(ctrl.dare.K).tolist()])
print("check           :", ctrl.check)

f, nis = zip(*ctrl.nis_log)
plt.plot(f, nis, "o-", color=BLUE, ms=4)
plt.axhline(1, color="k", ls="--", lw=1)
plt.axhline(ctrl.config.nis_threshold, color="0.5", ls=":", label="nis_threshold")
plt.xlabel("frame"); plt.ylabel("mean NIS / N_modes"); plt.legend(); plt.title("NIS monitor"); plt.show()

## 2.2 Changing conditions

The process noise triples at frame 6000 (seeing degrades). Configurations compared:

* adaptive (default): re-identify every 1000 frames
* fixed model: `max_updates=1` (identify once, as in `SimpleKalman.ipynb`)
* fixed model + `allow_integrator_fallback=True`: switch to the integrator when the NIS goes above `nis_threshold`
* adaptive + fallback: go back to Kalman when a new model is accepted
* adaptive with a forgetting factor instead of a window: `forgetting=1 - 1/1500`

In [ ]:
phi3, noise3 = ar1_modes(12000, A_SLOW, 0.1, 0.3, seed=4, q_change=(6000, 3.0))
runs = {
    "adaptive": toy_config(),
    "fixed model": toy_config(max_updates=1),
    "fixed + fallback": toy_config(max_updates=1, allow_integrator_fallback=True, nis_threshold=1.5),
    "adaptive + fallback": toy_config(allow_integrator_fallback=True, nis_threshold=1.5),
    "forgetting factor": toy_config(forgetting=1 - 1 / 1500),
}
results, ctrls = {}, {}
for name, cfg in runs.items():
    ctrls[name] = AdaptiveKalmanController(cfg, "cpu", DTYPE)
    results[name] = run_loop(ctrls[name], phi3, noise3)
results["integrator"] = run_loop(IntegratorController(4, 0.4, 0.99, dtype=DTYPE), phi3, noise3)

spans = [(3000, 6000), (6000, 7000), (7000, 12000)]
print("mean residual variance\n" + " " * 20 + "".join(f"{f'frames {a}-{b}':>19}" for a, b in spans))
for name, res in results.items():
    # res[j] is frame j + 2
    print(f"{name:>20}" + "".join(f"{float(res[a - 2:b - 2].pow(2).mean()):19.4f}" for a, b in spans))

for name, color in [("integrator", ORANGE), ("fixed model", AQUA), ("adaptive", BLUE)]:
    plt.plot(*rolling(results[name].pow(2).mean(1), 300), color=color, label=name)
plt.axvline(6000, color="k", lw=1)
plt.yscale("log"); plt.xlabel("frame"); plt.ylabel("residual variance (300-frame mean)"); plt.legend()
plt.title("Process noise x3 at frame 6000"); plt.show()

In [ ]:
for name in ["fixed model", "fixed + fallback", "adaptive + fallback"]:
    print(f"--- {name} (final mode: {ctrls[name].mode})")
    show_history(ctrls[name], skip=("accepted",) if name != "adaptive + fallback" else ())
    print()

* The fixed model keeps running with the old $Q$, and its NIS goes up (`nis_alarm`). With
  `allow_integrator_fallback=False` (the default), the alarm is only logged.
* `fixed + fallback` hands over to the integrator at frame 6200 and stays there, since nothing re-identifies (`max_updates=1`).
* The adaptive controller is **worse than both for about 1000 frames after the change**. The model it accepted at
  frame 6000 was identified on frames 4000-6000, before the change, and it came out with $a$ close to the cap and $q$
  10-30 times too small for three modes (a milder case of the problem in §2.4). With small gains, it reacts slowly to the
  stronger turbulence. From the next model on (frame 7000) its $Q$ is right, and it is the best of the three.
* `adaptive + fallback` catches that: the NIS jumps at frame 6200, the integrator takes over, and the model accepted at 7000 brings Kalman back.
* `nis_threshold` is a trade-off. Models from short windows can have a NIS of 1.5-2 without anything being badly wrong,
  so a low threshold makes the controller fall back more often than needed.

## 2.3 Vibrations with AR(2) modes

Modes 0 and 1 are vibrations (40 Hz and 25 Hz at 1 kHz, $\rho = 0.99$), modes 2 and 3 are turbulence.

In [ ]:
N = 12000
phi4, noise4 = ar1_modes(N, [0.99, 0.99, 0.99, 0.95], 0.1, 0.1)
phi4[:, 0] = vibration(N, 40, 0.99, 0.05, seed=3)
phi4[:, 1] = vibration(N, 25, 0.99, 0.05, seed=4)

ctrl_ar1 = AdaptiveKalmanController(toy_config(), "cpu", DTYPE)
ctrl_ar2 = AdaptiveKalmanController(toy_config(ar2_modes=(0, 1)), "cpu", DTYPE)
res_vib = {
    "integrator": run_loop(IntegratorController(4, 0.4, 0.99, dtype=DTYPE), phi4, noise4),
    "Kalman AR(1)": run_loop(ctrl_ar1, phi4, noise4),
    "Kalman AR(2) on modes 0, 1": run_loop(ctrl_ar2, phi4, noise4),
}
print("augmented state size:", ctrl_ar2.model.A.shape[0], "(4 modes + 2 previous values)")

colors = [ORANGE, AQUA, BLUE]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
width = 0.27
for j, ((name, res), color) in enumerate(zip(res_vib.items(), colors)):
    ax1.bar(torch.arange(4) + (j - 1) * width, res[4000:].pow(2).mean(0), width, color=color, label=name)
    seg = res[4000:11000, 0].reshape(-1, 1000)
    psd = (torch.fft.rfft(seg).abs() ** 2).mean(0)
    ax2.semilogy(torch.fft.rfftfreq(1000, 1 / 1000), psd, color=color, label=name)
ax1.set_xticks(range(4)); ax1.set_xlabel("mode"); ax1.set_ylabel("residual variance"); ax1.set_yscale("log")
ax1.legend(fontsize=8); ax1.set_title("Residual per mode (after frame 4000)")
ax2.set_xlabel("frequency (Hz)"); ax2.set_ylabel("PSD"); ax2.set_title("Residual PSD, mode 0 (40 Hz)")
plt.tight_layout(); plt.show()

## 2.4 Known failure mode: Q collapsing to 0 for one mode

**This is a limitation of the current identification.** On a 2000-frame window, the instrumental-variable
estimate of a slow mode sometimes exceeds 1. It is shrunk to `A_max_radius` = 0.9995, and the $Q$
estimated for that mode then comes out $\le 0$ and is clipped to 0. With $q = 0$ the model says the mode is
perfectly predictable, so $K = 0$ for it and the mode runs **open loop** until the next re-identification.
`validate_model` accepts the model because the rest of $Q$ is healthy.

On the realization below it happens on the very first model. To spot it, look for modes with $K_{ii} \approx 0$ and $a$ at the cap:

In [ ]:
class LoggingController(AdaptiveKalmanController):
    """Keeps a per-mode summary of every accepted model."""

    def reidentify(self):
        accepted = super().reidentify()
        if accepted:
            self.models = getattr(self, "models", []) + [(
                self.frame, torch.diagonal(self.model.A), torch.diagonal(self.model.Q), torch.diagonal(self.dare.K))]
        return accepted


phi5, noise5 = ar1_modes(12000, A_SLOW, 0.1, 0.3, seed=0)
ctrl5 = LoggingController(toy_config(), "cpu", DTYPE)
res5 = run_loop(ctrl5, phi5, noise5)
ctrl5_fb = AdaptiveKalmanController(toy_config(allow_integrator_fallback=True, nis_threshold=1.5), "cpu", DTYPE)
res5_fb = run_loop(ctrl5_fb, phi5, noise5)
res5_int = run_loop(IntegratorController(4, 0.4, 0.99, dtype=DTYPE), phi5, noise5)

print(f"{'frame':>6}  {'a (diag)':<32} {'q (diag)':<32} K (diag)")
for frame, a, q, K in ctrl5.models[:5]:
    fmt = lambda t, p: "[" + ", ".join(f"{v:.{p}f}" for v in t.tolist()) + "]"
    print(f"{frame:>6}  {fmt(a, 4):<32} {fmt(q, 4):<32} {fmt(K, 3)}")

for res, color, label in [(res5_int, ORANGE, "integrator"), (res5, BLUE, "adaptive Kalman"),
                          (res5_fb, AQUA, "adaptive Kalman + fallback")]:
    plt.plot(*rolling(res[:, 0].pow(2)), color=color, label=label)
event_lines(ctrl5)
plt.yscale("log"); plt.xlabel("frame"); plt.ylabel("residual variance, mode 0"); plt.legend()
plt.title("Q collapse on mode 0"); plt.show()

The NIS monitor does see it (the innovations of the open-loop mode are far larger than predicted), so
`allow_integrator_fallback=True` limits the damage. A proper fix belongs in `identify_model`/`validate_model`:
a per-mode floor on $Q$, or rejecting models where a mode has $Q_{ii} \approx 0$.

## 2.5 Real-time notes

* `step` returns a view on an internal buffer that the next call overwrites: `clone()` it if you keep it.
* All real-time state is preallocated and updated in place. `rt_step` never synchronizes with the host;
  `slow_path` does (identification, and reading the NIS accumulator every `nis_window` frames).
* **Double buffer**: `load_matrix` writes the new $M$ into the inactive buffer and flips `_active` between frames,
  so a frame never sees a half-written matrix. `load_matrix` also lets you run a model you built yourself (as `true_model_controller` does above).
* **CUDA graphs** (`use_cuda_graph=True`): the real-time step is captured once per buffer, so a frame is a single graph replay.

In [ ]:
if torch.cuda.is_available():
    for use_graph in (False, True):
        c = AdaptiveKalmanController(AdaptiveKalmanConfig(n_modes=150, max_updates=0, use_cuda_graph=use_graph), "cuda")
        z = torch.randn(150, device="cuda")
        for _ in range(200):
            c.rt_step(z)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(2000):
            c.rt_step(z)
        torch.cuda.synchronize()
        print(f"150 modes, use_cuda_graph={use_graph!s:<5}: {1e6 * (time.perf_counter() - t0) / 2000:6.1f} us per frame")
else:
    print("no CUDA device")

---
# Part 3: Closed loop with the pyramid WFS

Same simulation as `SimpleKalman.ipynb`, but:

* the Kalman controller starts from nothing: it runs the integrator during the first `window` frames, then
  identifies its own model from the closed-loop telemetry, and keeps re-identifying it;
* the integrator runs in the same loop, on the same atmosphere (each frame of `PhaseDataset` is used by both
  systems), so the comparison is frame by frame. Calling `dataset[0]` again would draw a new atmosphere.

> **Runtime.** Each frame simulates the WFS twice: about 1 minute for 8000 frames on the GPU used to write this notebook.

> **Result in the saved run: the adaptive Kalman does worse than the integrator.** See "Reading the results" at the end of this part.

In [ ]:
from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror
from AI4AO import imshow, imshow_multiple

device = "cuda"

In [ ]:
WFSParams = dict(
    {
        "Nres":30,
        "sampling":90/30,
        "D": 1.5,
        "centralObstruction": 0.,
        "useNoise": True,
        "Modulation": 0,
        "Wavelength": 635e-9,
        "Nphotons": [4.5, 6],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.05, 0.05],  # Fired parameter range (m)
        "L0": [10, 30.0000],  # Outter scale range (m)
        "Nphases": 1,  # Number of phases in the batch
        "Layers": [5, 10],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 2,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [0.9, 1.],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.]
    }
)

## Atmosphere parameters
DMParams = dict(
    {
        "Nactuator": 15,
        "Nmodes": 150,
        "FlipLeftRight": False,
        "FlipTopBottom": False,
        "moffatParam": 10,
        "signedAmplitude": 1e-5,
        "MechCoupling": 0.35
    }
)

In [ ]:
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()
dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()
M2C = dm.MakeZernikeM2C()

modes = dm(M2C.T)
wfs.BuildReconstructionMatrix(modes)

M2C_T = M2C.T
Nmodes = M2C.shape[1]
k_wave = 2 * torch.pi / dataset.wavelength

# true modal coefficients of an OPD map (least squares on the DM modes, inside the pupil),
# only used to diagnose the WFS below
in_pupil = dataset.pupil.flatten() > 0
modal_projector = torch.linalg.pinv(modes.reshape(Nmodes, -1)[:, in_pupil])   # [N_pix, Nmodes]

### Controller configuration

* `delay=2`: the loop below has the timing of §2 ($y_k = z_k + u_{k-2}$). `SimpleKalman.ipynb` pairs
  `z + z_estimated_kalman`, i.e. $u_{k-1}$; `pol_lag=1` reproduces that.
* `A_structure="block"` with Zernike radial orders: 150 modes and a 2000-frame window are too few for a
  full $150\times150$ $A$ (and $Q$, $R$ follow the same structure).
* Identification in float64 (slow path, once every `update_period` frames), real-time path in float32.
* The integrator uses the same gain and leak as the integrator it is compared with.

In [ ]:
gain, leak = 0.3, 0.995
n_frames = 8000

cfg = AdaptiveKalmanConfig(
    n_modes=Nmodes,
    delay=2,
    window=2000,                  # frames per identification (4 s at 500 Hz)
    update_period=1000,           # re-identify every 2 s
    A_method="instrumental_variable",
    A_structure="block",
    groups=zernike_radial_groups(Nmodes, device=device),
    A_max_radius=0.9995,
    integrator_gain=gain,
    integrator_leak=leak,
    ident_dtype=torch.float64,
    dare_dtype=torch.float64,
    # Other options to try:
    # A_structure="diagonal",                   # independent modes
    # max_updates=1,                            # identify once, then keep the model
    # forgetting=1 - 1 / 2000,                  # forgetting factor instead of a window
    # ar2_modes=(0, 1),                         # AR(2) for tip and tilt (vibrations)
    # R_fixed=estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, wfs.Nphotons, wfs.RON),
    # R_min_fraction=1e-2,                      # higher floor on R: less aggressive gains
    # allow_integrator_fallback=True,           # fall back to the integrator on NIS alarms
    # use_cuda_graph=True,                      # real-time step as a CUDA graph
)
psf_start = min(cfg.window + 1000, n_frames // 2)   # average the PSF once the first model has settled
diag_frames = range(500, min(2000, cfg.window))     # integrator frames used for the WFS diagnostic

In [ ]:
batch = dataset[0]                        # draws the atmosphere; later frames are dataset[i], i > 0
wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

ctrl = AdaptiveKalmanController(cfg, device, torch.float32)
systems = {
    "adaptive Kalman": ctrl,
    "integrator": IntegratorController(Nmodes, gain=gain, leak=leak, device=device),
}
loop = {name: dict(residual=torch.zeros_like(batch["opd"]),
                   u_prev=torch.zeros(1, Nmodes, device=device),
                   var=[], psf_sum=0.0, z=[], true=[])
        for name in systems}
input_var = []

t0 = time.perf_counter()
for i in range(n_frames):
    if i > 0:
        batch = dataset[i]
    opd_gt, pupil = batch["opd"], batch["pupil"]
    input_var.append(opd_gt[pupil > 0].var())

    for name, controller in systems.items():
        s = loop[name]
        wfs_frames = wfs(s["residual"], pupil)           # WFS sees the residual of the previous frame
        z = wfs.GetReconstructedOPD(wfs_frames)          # [1, Nmodes]
        if name == "integrator" and i in diag_frames:
            s["z"].append(z)
            s["true"].append(s["residual"].reshape(1, -1)[:, in_pupil] @ modal_projector)
        u = controller.step(z.reshape(-1)).clone().view(1, -1)

        s["residual"] = opd_gt - dm(s["u_prev"] @ M2C_T)   # the DM holds the previous command
        s["u_prev"] = u
        s["var"].append(s["residual"][pupil > 0].var())   # OPD variance inside the pupil (m^2)
        if i >= psf_start:
            s["psf_sum"] = s["psf_sum"] + wfs.GetPSF(s["residual"], pupil)

    if (i + 1) % 1000 == 0:
        print(f"frame {i + 1}/{n_frames}  mode={ctrl.mode:<9}  {time.perf_counter() - t0:6.0f} s")

input_rms_nm = 1e9 * torch.stack(input_var).sqrt().cpu()
for s in loop.values():
    s["rms_nm"] = 1e9 * torch.stack(s["var"]).sqrt().cpu()
    s["psf"] = (s["psf_sum"] / (n_frames - psf_start)).squeeze()

### Correction quality

Residual OPD RMS inside the pupil. The Strehl ratio (Maréchal, $e^{-\sigma^2}$) is also printed, but it is
close to 0 whenever the residual is above ~$\lambda/6$ = 100 nm.

In [ ]:
first = next((h["frame"] for h in ctrl.history if h["event"] == "accepted"), n_frames)
plt.axvspan(0, first, color="0.92", label="bootstrap (integrator)")
plt.plot(*rolling(input_rms_nm, 100), color="0.4", lw=1, label="no correction")
for name, color in [("integrator", ORANGE), ("adaptive Kalman", BLUE)]:
    plt.plot(*rolling(loop[name]["rms_nm"], 100), color=color, label=name)
event_lines(ctrl)
plt.yscale("log"); plt.xlabel("frame"); plt.ylabel("residual RMS (nm)"); plt.legend(fontsize=8)
plt.title("Residual OPD (100-frame mean, dotted: accepted models)"); plt.show()

print(f"{'':>16}  frames {psf_start}-{n_frames}:")
print(f"{'no correction':>16}: {float(input_rms_nm[psf_start:].mean()):7.0f} nm")
for name in systems:
    rms = loop[name]["rms_nm"][psf_start:]
    sr = torch.exp(-(k_wave * 1e-9 * rms) ** 2).mean()
    print(f"{name:>16}: {float(rms.mean()):7.0f} nm   Strehl {100 * float(sr):.1f}%")
print()
show_history(ctrl)

In [ ]:
if ctrl.nis_log:
    f, nis = zip(*ctrl.nis_log)
    plt.plot(f, nis, "o-", color=BLUE, ms=3)
    plt.axhline(1, color="k", ls="--", lw=1)
    plt.axhline(cfg.nis_threshold, color="0.5", ls=":", label="nis_threshold")
    plt.yscale("log"); plt.xlabel("frame"); plt.ylabel("mean NIS / N_modes"); plt.legend()
    plt.title("NIS monitor"); plt.show()

### Is the WFS linear? Optical gains

The pseudo-open-loop measurement $y_k = z_k + u_{k-2}$ assumes that $z$ measures the residual with unit gain.
During the bootstrap, we compare $z$ with the true modal coefficients of the residual (projection on the DM modes):
$g_i = \langle z_i, c_i \rangle / \langle c_i, c_i\rangle$. An unmodulated pyramid loses gain (and linearity)
as the residual grows. If $g \ll 1$, then $y_k \approx u_{k-2}$ carries almost no information about the atmosphere.

In [ ]:
Zd = torch.cat(loop["integrator"]["z"])
Td = torch.cat(loop["integrator"]["true"])
optical_gain = ((Zd * Td).sum(0) / (Td * Td).sum(0)).cpu()
correlation = ((Zd * Td).sum(0) / (Zd.norm(dim=0) * Td.norm(dim=0))).cpu()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
ax1.plot(optical_gain, ".", color=ORANGE); ax1.axhline(1, color="k", ls="--", lw=1)
ax1.set_xlabel("mode"); ax1.set_title("optical gain (integrator loop)")
ax2.plot(correlation, ".", color=ORANGE); ax2.set_ylim(0, 1.05)
ax2.set_xlabel("mode"); ax2.set_title("correlation of z with the true residual")
plt.tight_layout(); plt.show()
print(f"optical gain: median {float(optical_gain.median()):.2f}, "
      f"10-90% [{float(optical_gain.quantile(0.1)):.2f}, {float(optical_gain.quantile(0.9)):.2f}]")

### The identified model

The telemetry estimate of $R$ is compared with the photon + read-out-noise model used in `SimpleKalman.ipynb`
(`estimate_R`). The last lines check for the problems seen in §2.4: modes with $K_{ii} \approx 0$, and a
filter $F = (I - KH)A$ that is not stable.

In [ ]:
m = ctrl.model
if m is None:
    print("no model accepted:", [h.get("reasons") for h in ctrl.history])
else:
    R_photon = estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, wfs.Nphotons, wfs.RON)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    axes[0].plot(torch.diagonal(m.A_ar1).cpu(), ".", color=BLUE)
    axes[0].set_title("diagonal of A"); axes[0].set_xlabel("mode")
    axes[1].semilogy(torch.diagonal(R_photon).cpu(), ".", color=ORANGE, label="photon + RON model")
    axes[1].semilogy(torch.diagonal(m.R).cpu(), ".", color=BLUE, label="from telemetry")
    axes[1].set_title("diagonal of R"); axes[1].set_xlabel("mode"); axes[1].legend()
    axes[2].semilogy(torch.diagonal(m.Q).cpu().clamp(min=1e-12), ".", color=BLUE)
    axes[2].set_title("diagonal of Q"); axes[2].set_xlabel("mode")
    plt.tight_layout(); plt.show()

    imshow(m.A_ar1, colorbar=True)

    K_diag = torch.diagonal(ctrl.dare.K)[:Nmodes]
    I = torch.eye(m.A.shape[0], device=m.A.device, dtype=m.A.dtype)
    rho_F = float(torch.linalg.eigvals((I - ctrl.dare.K @ m.H) @ m.A).abs().max())
    print(f"DARE: {ctrl.dare.iterations} iterations, residual {ctrl.dare.residual:.1e}")
    print(f"K diagonal: median {float(K_diag.median()):.2f}")
    print("modes with K_ii ~ 0 (Q collapse):", torch.nonzero(K_diag.abs() < 1e-3).flatten().tolist() or "none")
    print(f"spectral radius of F = (I - KH)A: {rho_F:.5f}", "(unstable filter!)" if rho_F >= 1 else "")

### PSF and residuals

In [ ]:
psf_int, psf_kf = loop["integrator"]["psf"], loop["adaptive Kalman"]["psf"]
imshow(torch.log(torch.stack((psf_int, psf_kf))))

c = psf_kf.shape[-1] // 2
plt.semilogy(psf_int[c, :].cpu(), color=ORANGE, label="integrator")
plt.semilogy(psf_kf[c, :].cpu(), color=BLUE, label="adaptive Kalman")
plt.xlabel("pixel"); plt.ylabel("mean PSF"); plt.legend(); plt.title("Mean PSF, central row"); plt.show()

In [ ]:
fig, axes = imshow_multiple(
    [
        {"tensor": opd_gt, "title": "Input OPD", "same_scale": True},
        {"tensor": loop["integrator"]["residual"], "title": "Residual, integrator", "scale_reference": opd_gt},
        {"tensor": loop["adaptive Kalman"]["residual"], "title": "Residual, Kalman", "scale_reference": opd_gt},
    ],
    max_channel_number = 9
)

### Reading the results

In the run saved in this notebook, the adaptive Kalman is **worse than the integrator**, and its NIS stays far above 1
(the models do not describe the telemetry). Two things contribute:

1. **The WFS is not linear at `r0 = 0.05`.** The optical gains of the unmodulated pyramid are around 0.1 while
   the integrator runs, and the integrator only reduces the residual by ~20-25% compared with no correction
   (see the printout above; $\lambda$ = 635 nm).
   With $g \approx 0.1$, the pseudo-open-loop telemetry is mostly the command history, so there is little to identify.
2. **The identified models are fragile, even where the WFS is linear.** In separate runs (not saved here) with `r0 = 0.12` (the value in the committed
   `SimpleKalman.ipynb`), the optical gains are ~0.78 and the integrator reaches ~65 nm. The adaptive Kalman still
   diverges (>1000 nm with the block structure, ~230 nm with the diagonal one). The first model has
   $R$ at the `R_min_fraction` floor for many modes, so $K \approx 0.9$ (a very aggressive controller). One mode has $Q \approx 0$ (the
   §2.4 problem), and $F = (I-KH)A$ has a spectral radius just above 1. Once one mode drifts, the pyramid loses
   gain for all modes. The pseudo-open-loop controller has no integral action to pull the loop back, while the integrator does.
   Taking $R$ from the photon model does not fix it. A higher `R_min_fraction` or a 10x larger $R$ only
   "works" because no model is accepted and the integrator keeps running.

So the controller mechanics work (Parts 1-2), but the identification needs to be more robust before it can beat the
integrator in this loop: a per-mode floor on $Q$, rejecting models with an unstable $F$, a more conservative $R$, and
compensating (or at least monitoring) the optical gains.